In [4]:
from pathlib import Path
from functools import reduce

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

base_path = Path(r"C:\Users\selim\Desktop\Motosport_Data_Platform\data_lake\gold\fastf1")

gold_tables = [
    "gold_constructor_recent_form",
    "gold_driver_recent_form",
    "gold_lap_prediction_dataset",
]

def read_all_seasons_for_table(table_name: str):
    dfs = []

    print(f"\n===== Loading {table_name} =====")

    for season_dir in sorted(base_path.iterdir()):
        if not season_dir.is_dir():
            continue

        table_path = season_dir / table_name

        if not table_path.exists():
            print(f"Skipping {season_dir.name}: table not found")
            continue

        print(f"Loading {season_dir.name}")
        df = spark.read.parquet(str(table_path))
        dfs.append(df)

    if not dfs:
        raise ValueError(f"No data found for table: {table_name}")

    return reduce(
        lambda a, b: a.unionByName(b, allowMissingColumns=True),
        dfs
    )

gold_constructor_recent_form_df = read_all_seasons_for_table("gold_constructor_recent_form")
gold_driver_recent_form_df = read_all_seasons_for_table("gold_driver_recent_form")
gold_lap_prediction_dataset_df = read_all_seasons_for_table("gold_lap_prediction_dataset")



===== Loading gold_constructor_recent_form =====
Loading season_2018
Loading season_2019
Loading season_2020
Loading season_2021
Loading season_2022
Loading season_2023
Loading season_2024
Loading season_2025

===== Loading gold_driver_recent_form =====
Loading season_2018
Loading season_2019
Loading season_2020
Loading season_2021
Loading season_2022
Loading season_2023
Loading season_2024
Loading season_2025

===== Loading gold_lap_prediction_dataset =====
Loading season_2018
Loading season_2019
Loading season_2020
Loading season_2021
Loading season_2022
Loading season_2023
Loading season_2024
Loading season_2025


In [5]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


print("\n===== FINAL COUNTS =====")
print("gold_constructor_recent_form:", gold_constructor_recent_form_df.count())
print("gold_driver_recent_form:", gold_driver_recent_form_df.count())
print("gold_lap_prediction_dataset:", gold_lap_prediction_dataset_df.count())

print("\n===== HEADS =====")

print("\n gold_constructor_recent_form_df:")
display(gold_constructor_recent_form_df.limit(5).toPandas())

print("\n gold_driver_recent_form_df:")
display(gold_driver_recent_form_df.limit(5).toPandas())

print("\n gold_lap_prediction_dataset_df:")
display(gold_lap_prediction_dataset_df.limit(5).toPandas())


===== FINAL COUNTS =====
gold_constructor_recent_form: 1529
gold_driver_recent_form: 2981
gold_lap_prediction_dataset: 164699

===== HEADS =====

 gold_constructor_recent_form_df:


,season,race,team_name,race_date,num_drivers,avg_finish_position,best_finish_position,total_position_gain,avg_lap_time_ms,best_lap_time_ms,total_pit_stops,top_10_finishes,podium_finishes,dnf_count,avg_finish_last_3,avg_finish_last_5,avg_position_gain_last_5,top_10_count_last_5,podium_count_last_5,dnf_count_last_5
0,2018,australia,Ferrari,2018-03-25 05:13:19.169,2,2.0,1,1,92697.198276,86373.0,4,2,2,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,china,Ferrari,2018-04-15 06:13:36.058,2,5.5,3,-8,101290.851852,96456.0,4,2,1,0,2.000000,2.00,1.0,2.0,2.0,0.0
2,2018,azerbaijan,Ferrari,2018-04-29 12:13:42.330,2,3.0,2,1,110025.023171,105530.0,5,2,1,0,3.750000,3.75,-3.5,4.0,3.0,0.0
3,2018,spain,Ferrari,2018-05-13 13:13:11.037,2,10.5,4,-14,90061.171154,79128.0,5,1,0,1,3.500000,3.50,-2.0,6.0,4.0,0.0
4,2018,monaco,Ferrari,2018-05-27 13:12:47.349,2,3.0,2,0,79327.416667,76065.0,4,2,1,0,6.333333,5.25,-5.0,7.0,4.0,1.0



 gold_driver_recent_form_df:


,season,race,driver,team_name,race_date,laps_completed,avg_lap_time_ms,best_lap_time_ms,pit_stop_count,grid_position,finish_position,race_status,position_gain,finished_top_10,finished_top_3,dnf_flag,avg_finish_last_3,avg_finish_last_5,avg_position_gain_last_5,top_10_count_last_5,podium_count_last_5,dnf_count_last_5
0,2018,australia,ALO,McLaren,2018-03-25 05:13:19.169,58,93123.603448,86978.0,2,10,5,Finished,5,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,china,ALO,McLaren,2018-04-15 06:13:36.058,56,102982.482143,97234.0,2,13,7,Finished,6,1,0,0,5.000000,5.000000,5.000000,1.0,0.0,0.0
2,2018,azerbaijan,ALO,McLaren,2018-04-29 12:13:42.330,41,112774.536585,107449.0,1,12,7,Finished,5,1,0,0,6.000000,6.000000,5.500000,2.0,0.0,0.0
3,2018,spain,ALO,McLaren,2018-05-13 13:13:11.037,64,88010.062500,80727.0,2,8,8,+1 Lap,0,1,0,1,6.333333,6.333333,5.333333,3.0,0.0,0.0
4,2018,monaco,ALO,McLaren,2018-05-27 13:12:47.349,52,79687.980769,77018.0,2,7,20,Gearbox,-13,0,0,1,7.333333,6.750000,4.000000,4.0,0.0,1.0



 gold_lap_prediction_dataset_df:


,season,race,driver,driver_number,team_name,lap_number,lapstartdate,position,stint,compound,tyre_life,fresh_tyre,is_pit_lap,is_pit_in_lap,is_pit_out_lap,is_green,is_yellow,is_safety_car,is_red_flag,is_vsc,air_temp,humidity,pressure,track_temp,wind_direction,wind_speed,is_raining,last_lap_time_ms,last_lap_is_green,last_lap_is_yellow,last_lap_is_safety_car,avg_lap_time_last_2,avg_lap_time_last_3,target_lap_time_ms
0,2018,australia,ALO,14,McLaren,1,2018-03-25 05:13:19.169,10,NaN,nan,NaN,True,False,False,False,False,False,False,False,False,24.2,36.3,997.0,38.9,307.0,4.1,False,NaN,None,None,None,NaN,NaN,101528.0
1,2018,australia,ALO,14,McLaren,2,2018-03-25 05:15:00.875,10,1.0,ULTRASOFT,1.0,True,False,False,False,False,False,False,False,False,24.2,36.3,996.9,38.2,296.0,3.8,False,101528.0,False,False,False,101528.0,101528.0,91565.0
2,2018,australia,ALO,14,McLaren,3,2018-03-25 05:16:32.440,10,1.0,ULTRASOFT,2.0,True,False,False,False,False,False,False,False,False,23.9,36.5,997.1,36.7,289.0,4.3,False,91565.0,False,False,False,96546.5,96546.5,91304.0
3,2018,australia,ALO,14,McLaren,4,2018-03-25 05:18:03.744,10,1.0,ULTRASOFT,3.0,True,False,False,False,False,False,False,False,False,23.9,36.3,997.1,36.8,255.0,2.9,False,91304.0,False,False,False,91434.5,94799.0,90551.0
4,2018,australia,ALO,14,McLaren,5,2018-03-25 05:19:34.295,10,1.0,ULTRASOFT,4.0,True,False,False,False,False,False,False,False,False,23.5,36.3,997.2,36.4,267.0,2.5,False,90551.0,False,False,False,90927.5,91140.0,91910.0


In [6]:
driver_cols_to_rename = [
    "avg_finish_last_3",
    "avg_finish_last_5",
    "avg_position_gain_last_5",
    "top_10_count_last_5",
    "podium_count_last_5",
    "dnf_count_last_5",
]

for col_name in driver_cols_to_rename:
    gold_driver_recent_form_df = gold_driver_recent_form_df.withColumnRenamed(
        col_name,
        f"driver_{col_name}"
    )

In [7]:
constructor_cols_to_rename = [
    "avg_finish_last_3",
    "avg_finish_last_5",
    "avg_position_gain_last_5",
    "top_10_count_last_5",
    "podium_count_last_5",
    "dnf_count_last_5",
]

for col_name in constructor_cols_to_rename:
    gold_constructor_recent_form_df = gold_constructor_recent_form_df.withColumnRenamed(
        col_name,
        f"constructor_{col_name}"
    )

In [8]:
print("\n gold_constructor_recent_form_df:")
display(gold_constructor_recent_form_df.limit(5).toPandas())

print("\n gold_driver_recent_form_df:")
display(gold_driver_recent_form_df.limit(5).toPandas())


 gold_constructor_recent_form_df:


,season,race,team_name,race_date,num_drivers,avg_finish_position,best_finish_position,total_position_gain,avg_lap_time_ms,best_lap_time_ms,total_pit_stops,top_10_finishes,podium_finishes,dnf_count,constructor_avg_finish_last_3,constructor_avg_finish_last_5,constructor_avg_position_gain_last_5,constructor_top_10_count_last_5,constructor_podium_count_last_5,constructor_dnf_count_last_5
0,2018,australia,Ferrari,2018-03-25 05:13:19.169,2,2.0,1,1,92697.198276,86373.0,4,2,2,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,china,Ferrari,2018-04-15 06:13:36.058,2,5.5,3,-8,101290.851852,96456.0,4,2,1,0,2.000000,2.00,1.0,2.0,2.0,0.0
2,2018,azerbaijan,Ferrari,2018-04-29 12:13:42.330,2,3.0,2,1,110025.023171,105530.0,5,2,1,0,3.750000,3.75,-3.5,4.0,3.0,0.0
3,2018,spain,Ferrari,2018-05-13 13:13:11.037,2,10.5,4,-14,90061.171154,79128.0,5,1,0,1,3.500000,3.50,-2.0,6.0,4.0,0.0
4,2018,monaco,Ferrari,2018-05-27 13:12:47.349,2,3.0,2,0,79327.416667,76065.0,4,2,1,0,6.333333,5.25,-5.0,7.0,4.0,1.0



 gold_driver_recent_form_df:


,season,race,driver,team_name,race_date,laps_completed,avg_lap_time_ms,best_lap_time_ms,pit_stop_count,grid_position,finish_position,race_status,position_gain,finished_top_10,finished_top_3,dnf_flag,driver_avg_finish_last_3,driver_avg_finish_last_5,driver_avg_position_gain_last_5,driver_top_10_count_last_5,driver_podium_count_last_5,driver_dnf_count_last_5
0,2018,australia,ALO,McLaren,2018-03-25 05:13:19.169,58,93123.603448,86978.0,2,10,5,Finished,5,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,china,ALO,McLaren,2018-04-15 06:13:36.058,56,102982.482143,97234.0,2,13,7,Finished,6,1,0,0,5.000000,5.000000,5.000000,1.0,0.0,0.0
2,2018,azerbaijan,ALO,McLaren,2018-04-29 12:13:42.330,41,112774.536585,107449.0,1,12,7,Finished,5,1,0,0,6.000000,6.000000,5.500000,2.0,0.0,0.0
3,2018,spain,ALO,McLaren,2018-05-13 13:13:11.037,64,88010.062500,80727.0,2,8,8,+1 Lap,0,1,0,1,6.333333,6.333333,5.333333,3.0,0.0,0.0
4,2018,monaco,ALO,McLaren,2018-05-27 13:12:47.349,52,79687.980769,77018.0,2,7,20,Gearbox,-13,0,0,1,7.333333,6.750000,4.000000,4.0,0.0,1.0


In [9]:

driver_recent_features = [
    "season",
    "race",
    "driver",
    "driver_avg_finish_last_3",
    "driver_avg_finish_last_5",
    "driver_avg_position_gain_last_5",
    "driver_top_10_count_last_5",
    "driver_podium_count_last_5",
    "driver_dnf_count_last_5",
]

constructor_recent_features = [
    "season",
    "race",
    "team_name",
    "constructor_avg_finish_last_3",
    "constructor_avg_finish_last_5",
    "constructor_avg_position_gain_last_5",
    "constructor_top_10_count_last_5",
    "constructor_podium_count_last_5",
    "constructor_dnf_count_last_5",
]

driver_recent_df = gold_driver_recent_form_df.select(*driver_recent_features)

constructor_recent_df = gold_constructor_recent_form_df.select(*constructor_recent_features)

ml_feature_df = (
    gold_lap_prediction_dataset_df
    .join(
        driver_recent_df,
        on=["season", "race", "driver"],
        how="left"
    )
    .join(
        constructor_recent_df,
        on=["season", "race", "team_name"],
        how="left"
    )
)

print("Rows:", ml_feature_df.count())
print("Columns:", len(ml_feature_df.columns))

display(ml_feature_df.limit(10).toPandas())

Rows: 164699
Columns: 46


,season,race,team_name,driver,driver_number,lap_number,lapstartdate,position,stint,compound,tyre_life,fresh_tyre,is_pit_lap,is_pit_in_lap,is_pit_out_lap,is_green,is_yellow,is_safety_car,is_red_flag,is_vsc,air_temp,humidity,pressure,track_temp,wind_direction,wind_speed,is_raining,last_lap_time_ms,last_lap_is_green,last_lap_is_yellow,last_lap_is_safety_car,avg_lap_time_last_2,avg_lap_time_last_3,target_lap_time_ms,driver_avg_finish_last_3,driver_avg_finish_last_5,driver_avg_position_gain_last_5,driver_top_10_count_last_5,driver_podium_count_last_5,driver_dnf_count_last_5,constructor_avg_finish_last_3,constructor_avg_finish_last_5,constructor_avg_position_gain_last_5,constructor_top_10_count_last_5,constructor_podium_count_last_5,constructor_dnf_count_last_5
0,2018,australia,McLaren,ALO,14,1,2018-03-25 05:13:19.169,10,NaN,nan,NaN,True,False,False,False,False,False,False,False,False,24.2,36.3,997.0,38.9,307.0,4.1,False,NaN,None,None,None,NaN,NaN,101528.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,australia,McLaren,ALO,14,2,2018-03-25 05:15:00.875,10,1.0,ULTRASOFT,1.0,True,False,False,False,False,False,False,False,False,24.2,36.3,996.9,38.2,296.0,3.8,False,101528.0,False,False,False,101528.0,101528.000000,91565.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018,australia,McLaren,ALO,14,3,2018-03-25 05:16:32.440,10,1.0,ULTRASOFT,2.0,True,False,False,False,False,False,False,False,False,23.9,36.5,997.1,36.7,289.0,4.3,False,91565.0,False,False,False,96546.5,96546.500000,91304.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018,australia,McLaren,ALO,14,4,2018-03-25 05:18:03.744,10,1.0,ULTRASOFT,3.0,True,False,False,False,False,False,False,False,False,23.9,36.3,997.1,36.8,255.0,2.9,False,91304.0,False,False,False,91434.5,94799.000000,90551.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018,australia,McLaren,ALO,14,5,2018-03-25 05:19:34.295,10,1.0,ULTRASOFT,4.0,True,False,False,False,False,False,False,False,False,23.5,36.3,997.2,36.4,267.0,2.5,False,90551.0,False,False,False,90927.5,91140.000000,91910.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2018,australia,McLaren,ALO,14,6,2018-03-25 05:21:06.205,10,1.0,ULTRASOFT,5.0,True,False,False,False,False,False,False,False,False,23.4,36.5,997.2,36.3,283.0,2.1,False,91910.0,False,False,False,91230.5,91255.000000,91731.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2018,australia,McLaren,ALO,14,7,2018-03-25 05:22:37.936,10,1.0,ULTRASOFT,6.0,True,False,False,False,False,False,False,False,False,23.5,36.7,997.1,36.4,301.0,3.4,False,91731.0,False,False,False,91820.5,91397.333333,91167.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2018,australia,McLaren,ALO,14,8,2018-03-25 05:24:09.103,10,1.0,ULTRASOFT,7.0,True,False,False,False,True,False,False,False,False,23.9,35.4,997.0,37.6,334.0,4.1,False,91167.0,False,False,False,91449.0,91602.666667,90926.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2018,australia,McLaren,ALO,14,9,2018-03-25 05:25:40.029,10,1.0,ULTRASOFT,8.0,True,False,False,False,True,False,False,False,False,23.9,35.9,997.0,38.0,294.0,2.0,False,90926.0,True,False,False,91046.5,91274.666667,90407.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2018,australia,McLaren,ALO,14,10,2018-03-25 05:27:10.436,10,1.0,ULTRASOFT,9.0,True,False,False,False,False,False,False,False,False,24.0,35.1,997.0,38.4,326.0,3.3,False,90407.0,True,False,False,90666.5,90833.333333,90985.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
from pyspark.sql.functions import col, when
from pyspark.sql.window import Window
from pyspark.sql.functions import first

# Tyre life is 0 when null (happens only on race start)
ml_feature_df = ml_feature_df.withColumn(
    "tyre_life",
    when(col("tyre_life").isNull(), 0).otherwise(col("tyre_life"))
)

#change compound nan to next lap compound (happens only on lap 1)
ml_feature_df = ml_feature_df.withColumn(
    "compound",
    when(col("compound") == "nan", None).otherwise(col("compound"))
)

window_next = Window.partitionBy("season", "race", "driver") \
    .orderBy(col("lap_number").asc()) \
    .rowsBetween(0, Window.unboundedFollowing)

ml_feature_df = ml_feature_df.withColumn(
    "compound",
    first("compound", ignorenulls=True).over(window_next)
)
ml_feature_df = ml_feature_df.filter(col("compound").isNotNull())

# Stint is 1 when null (happens only on race start)
ml_feature_df = ml_feature_df.withColumn(
    "stint",
    when((col("stint").isNull()), 1)
    .otherwise(col("stint"))
) 
# Change boolean to 0 and 1

bool_cols = [
    "fresh_tyre", "is_pit_lap", "is_pit_in_lap", "is_pit_out_lap",
    "is_green", "is_yellow", "is_safety_car", "is_red_flag", "is_vsc",
    "last_lap_is_green", "last_lap_is_yellow", "last_lap_is_safety_car", 
    "is_raining"
]

for c in bool_cols:
    ml_feature_df = ml_feature_df.withColumn(c, col(c).cast("int"))


display(ml_feature_df.limit(10).toPandas())

,season,race,team_name,driver,driver_number,lap_number,lapstartdate,position,stint,compound,tyre_life,fresh_tyre,is_pit_lap,is_pit_in_lap,is_pit_out_lap,is_green,is_yellow,is_safety_car,is_red_flag,is_vsc,air_temp,humidity,pressure,track_temp,wind_direction,wind_speed,is_raining,last_lap_time_ms,last_lap_is_green,last_lap_is_yellow,last_lap_is_safety_car,avg_lap_time_last_2,avg_lap_time_last_3,target_lap_time_ms,driver_avg_finish_last_3,driver_avg_finish_last_5,driver_avg_position_gain_last_5,driver_top_10_count_last_5,driver_podium_count_last_5,driver_dnf_count_last_5,constructor_avg_finish_last_3,constructor_avg_finish_last_5,constructor_avg_position_gain_last_5,constructor_top_10_count_last_5,constructor_podium_count_last_5,constructor_dnf_count_last_5
0,2018,australia,Renault,SAI,55,1,2018-03-25 05:13:19.169,9,1,ULTRASOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,NaN,NaN,NaN,NaN,NaN,NaN,100565.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,australia,Renault,SAI,55,2,2018-03-25 05:14:59.912,9,1,ULTRASOFT,4,0,0,0,0,0,0,0,0,0,24.2,36.3,996.9,38.2,296.0,3.8,0,100565.0,0.0,0.0,0.0,100565.0,100565.000000,91322.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018,australia,Renault,SAI,55,3,2018-03-25 05:16:31.234,9,1,ULTRASOFT,5,0,0,0,0,0,0,0,0,0,23.9,36.5,997.1,36.7,289.0,4.3,0,91322.0,0.0,0.0,0.0,95943.5,95943.500000,91267.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018,australia,Renault,SAI,55,4,2018-03-25 05:18:02.501,9,1,ULTRASOFT,6,0,0,0,0,0,0,0,0,0,23.9,36.3,997.1,36.8,255.0,2.9,0,91267.0,0.0,0.0,0.0,91294.5,94384.666667,90672.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018,australia,Renault,SAI,55,5,2018-03-25 05:19:33.173,9,1,ULTRASOFT,7,0,0,0,0,0,1,0,0,0,23.5,36.3,997.2,36.4,267.0,2.5,0,90672.0,0.0,0.0,0.0,90969.5,91087.000000,91834.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2018,australia,Renault,SAI,55,6,2018-03-25 05:21:05.007,9,1,ULTRASOFT,8,0,0,0,0,0,0,0,0,0,23.4,36.5,997.2,36.3,283.0,2.1,0,91834.0,0.0,1.0,0.0,91253.0,91257.666667,91816.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2018,australia,Renault,SAI,55,7,2018-03-25 05:22:36.823,9,1,ULTRASOFT,9,0,0,0,0,0,0,0,0,0,23.5,36.7,997.1,36.4,301.0,3.4,0,91816.0,0.0,0.0,0.0,91825.0,91440.666667,90680.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2018,australia,Renault,SAI,55,8,2018-03-25 05:24:07.503,9,1,ULTRASOFT,10,0,0,0,0,1,0,0,0,0,23.8,35.6,997.0,37.5,297.0,2.9,0,90680.0,0.0,0.0,0.0,91248.0,91443.333333,90596.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2018,australia,Renault,SAI,55,9,2018-03-25 05:25:38.099,9,1,ULTRASOFT,11,0,0,0,0,1,0,0,0,0,23.9,35.9,997.0,38.0,294.0,2.0,0,90596.0,1.0,0.0,0.0,90638.0,91030.666667,90239.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2018,australia,Renault,SAI,55,10,2018-03-25 05:27:08.338,9,1,ULTRASOFT,12,0,0,0,0,0,0,0,0,0,24.1,35.1,997.1,38.0,283.0,4.0,0,90239.0,1.0,0.0,0.0,90417.5,90505.000000,92039.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = ml_feature_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ml_feature_df.columns
])

display(null_counts.toPandas())

,season,race,team_name,driver,driver_number,lap_number,lapstartdate,position,stint,compound,tyre_life,fresh_tyre,is_pit_lap,is_pit_in_lap,is_pit_out_lap,is_green,is_yellow,is_safety_car,is_red_flag,is_vsc,air_temp,humidity,pressure,track_temp,wind_direction,wind_speed,is_raining,last_lap_time_ms,last_lap_is_green,last_lap_is_yellow,last_lap_is_safety_car,avg_lap_time_last_2,avg_lap_time_last_3,target_lap_time_ms,driver_avg_finish_last_3,driver_avg_finish_last_5,driver_avg_position_gain_last_5,driver_top_10_count_last_5,driver_podium_count_last_5,driver_dnf_count_last_5,constructor_avg_finish_last_3,constructor_avg_finish_last_5,constructor_avg_position_gain_last_5,constructor_top_10_count_last_5,constructor_podium_count_last_5,constructor_dnf_count_last_5
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2980,2980,2980,2980,2980,2980,0,9111,9111,9111,9111,9111,9111,8276,8276,8276,8276,8276,8276


In [12]:
# Add a feature column to indicate if past values are missing for rolling averages 

ml_feature_df = ml_feature_df.withColumn(
    "driver_history_available",
    when(col("driver_avg_finish_last_3").isNull(), 0).otherwise(1)
)

ml_feature_df = ml_feature_df.withColumn(
    "constructor_history_available",
    when(col("constructor_avg_finish_last_3").isNull(), 0).otherwise(1)
)

ml_feature_df = ml_feature_df.withColumn(
    "last_lap_history_available",
    when(col("last_lap_time_ms").isNull(), 0).otherwise(1)
)

# Replace nulls in rolling average features with a default value
ml_feature_df = ml_feature_df.fillna({
    "driver_avg_finish_last_3": 10,
    "driver_avg_finish_last_5": 10,
    "driver_avg_position_gain_last_5": 0,
    "driver_top_10_count_last_5": 0,
    "driver_podium_count_last_5": 0,
    "driver_dnf_count_last_5": 0,

    "constructor_avg_finish_last_3": 10,
    "constructor_avg_finish_last_5": 10,
    "constructor_avg_position_gain_last_5": 0,
    "constructor_top_10_count_last_5": 0,
    "constructor_podium_count_last_5": 0,
    "constructor_dnf_count_last_5": 0,

    "last_lap_time_ms": 0,
    "last_lap_is_green": 0,
    "last_lap_is_yellow": 0,
    "last_lap_is_safety_car": 0,
    "avg_lap_time_last_2": 0,
    "avg_lap_time_last_3": 0,
})


In [13]:
null_counts = ml_feature_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ml_feature_df.columns
])

display(null_counts.toPandas())

,season,race,team_name,driver,driver_number,lap_number,lapstartdate,position,stint,compound,tyre_life,fresh_tyre,is_pit_lap,is_pit_in_lap,is_pit_out_lap,is_green,is_yellow,is_safety_car,is_red_flag,is_vsc,air_temp,humidity,pressure,track_temp,wind_direction,wind_speed,is_raining,last_lap_time_ms,last_lap_is_green,last_lap_is_yellow,last_lap_is_safety_car,avg_lap_time_last_2,avg_lap_time_last_3,target_lap_time_ms,driver_avg_finish_last_3,driver_avg_finish_last_5,driver_avg_position_gain_last_5,driver_top_10_count_last_5,driver_podium_count_last_5,driver_dnf_count_last_5,constructor_avg_finish_last_3,constructor_avg_finish_last_5,constructor_avg_position_gain_last_5,constructor_top_10_count_last_5,constructor_podium_count_last_5,constructor_dnf_count_last_5,driver_history_available,constructor_history_available,last_lap_history_available
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [14]:
# Sort the final dataset to have chronological order
ml_feature_df = ml_feature_df.orderBy(
    col("lapstartdate"),
    col("driver"),
    col("lap_number")
)
 
# Drop columns that won't be used in modeling
ml_feature_df = ml_feature_df.drop("lapstartdate", "driver_number")
display(ml_feature_df.limit(10).toPandas())

,season,race,team_name,driver,lap_number,position,stint,compound,tyre_life,fresh_tyre,is_pit_lap,is_pit_in_lap,is_pit_out_lap,is_green,is_yellow,is_safety_car,is_red_flag,is_vsc,air_temp,humidity,pressure,track_temp,wind_direction,wind_speed,is_raining,last_lap_time_ms,last_lap_is_green,last_lap_is_yellow,last_lap_is_safety_car,avg_lap_time_last_2,avg_lap_time_last_3,target_lap_time_ms,driver_avg_finish_last_3,driver_avg_finish_last_5,driver_avg_position_gain_last_5,driver_top_10_count_last_5,driver_podium_count_last_5,driver_dnf_count_last_5,constructor_avg_finish_last_3,constructor_avg_finish_last_5,constructor_avg_position_gain_last_5,constructor_top_10_count_last_5,constructor_podium_count_last_5,constructor_dnf_count_last_5,driver_history_available,constructor_history_available,last_lap_history_available
0,2018,australia,McLaren,ALO,1,10,1,ULTRASOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,101528.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
1,2018,australia,Mercedes,BOT,1,15,1,ULTRASOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,103824.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
2,2018,australia,Sauber,ERI,1,16,1,SUPERSOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,104674.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
3,2018,australia,Toro Rosso,GAS,1,17,1,ULTRASOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,105060.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
4,2018,australia,Haas F1 Team,GRO,1,6,1,ULTRASOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,98450.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
5,2018,australia,Mercedes,HAM,1,1,1,ULTRASOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,94233.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
6,2018,australia,Toro Rosso,HAR,1,20,1,SOFT,0,1,1,1,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,125584.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
7,2018,australia,Renault,HUL,1,7,1,ULTRASOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,99077.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
8,2018,australia,Sauber,LEC,1,18,1,SUPERSOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,105584.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0
9,2018,australia,Haas F1 Team,MAG,1,4,1,ULTRASOFT,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,97107.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0


In [15]:
import pandas as pd

ml_pd_feature_df = ml_feature_df.toPandas()

# One-hot encode categorical variables
categorical_cols = ["race", "team_name", "driver", "compound"]
ml_pd_feature_df = pd.get_dummies(ml_pd_feature_df, columns=categorical_cols)

In [16]:
ml_pd_feature_df.head()

,season,lap_number,position,stint,tyre_life,fresh_tyre,is_pit_lap,is_pit_in_lap,is_pit_out_lap,is_green,is_yellow,is_safety_car,is_red_flag,is_vsc,air_temp,humidity,pressure,track_temp,wind_direction,wind_speed,is_raining,last_lap_time_ms,last_lap_is_green,last_lap_is_yellow,last_lap_is_safety_car,avg_lap_time_last_2,avg_lap_time_last_3,target_lap_time_ms,driver_avg_finish_last_3,driver_avg_finish_last_5,driver_avg_position_gain_last_5,driver_top_10_count_last_5,driver_podium_count_last_5,driver_dnf_count_last_5,constructor_avg_finish_last_3,constructor_avg_finish_last_5,constructor_avg_position_gain_last_5,constructor_top_10_count_last_5,constructor_podium_count_last_5,constructor_dnf_count_last_5,driver_history_available,constructor_history_available,last_lap_history_available,race_abu_dhabi,race_australia,race_austria,race_azerbaijan,race_bahrain,race_belgium,race_brazil,race_canada,race_china,race_france,race_germany,race_great_britain,race_hungary,race_italy,race_japan,race_mexico,race_monaco,race_netherlands,race_portugal,race_qatar,race_russia,race_saudi_arabia,race_singapore,race_spain,race_turkey,race_united_arab_emirates,race_united_kingdom,race_united_states,team_name_Alfa Romeo,team_name_Alfa Romeo Racing,team_name_AlphaTauri,team_name_Alpine,team_name_Aston Martin,team_name_Ferrari,team_name_Force India,team_name_Haas F1 Team,team_name_Kick Sauber,team_name_McLaren,team_name_Mercedes,team_name_RB,team_name_Racing Bulls,team_name_Racing Point,team_name_Red Bull Racing,team_name_Renault,team_name_Sauber,team_name_Toro Rosso,team_name_Williams,driver_ALB,driver_ALO,driver_ANT,driver_BEA,driver_BOR,driver_BOT,driver_COL,driver_DEV,driver_DOO,driver_ERI,driver_FIT,driver_GAS,driver_GIO,driver_GRO,driver_HAD,driver_HAM,driver_HAR,driver_HUL,driver_KUB,driver_KVY,driver_LAT,driver_LAW,driver_LEC,driver_MAG,driver_MAZ,driver_MSC,driver_NOR,driver_OCO,driver_PER,driver_PIA,driver_RAI,driver_RIC,driver_RUS,driver_SAI,driver_SAR,driver_SIR,driver_STR,driver_TSU,driver_VAN,driver_VER,driver_VET,driver_ZHO,compound_HARD,compound_HYPERSOFT,compound_INTERMEDIATE,compound_MEDIUM,compound_None,compound_SOFT,compound_SUPERSOFT,compound_ULTRASOFT,compound_UNKNOWN,compound_WET
0,2018,1,10,1,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,101528.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
1,2018,1,15,1,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,103824.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,2018,1,16,1,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,104674.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,